# Módulo 06 · Aula 04 — Segurança e Filtros

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"A API está no ar e funcionando. Só que **qualquer pessoa** com a URL pode apagar um produto. O estagiário do marketing descobriu o `/docs` e zerou o estoque 'pra testar'. E a senha do banco está escrita no `config.py`, que está no GitHub."*
> — Sua chefe, com razão

Três buracos:

1. **Sem autenticação** — a API não sabe quem está chamando.
2. **Sem autorização** — mesmo sabendo, não decide o que cada um pode.
3. **Segredo no código** — a senha do banco versionada.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | `Settings` por ambiente | 🔴 Tirar segredo do código |
| 2 | Hash de senha | 🔴 Nunca guardar senha |
| 3 | JWT | Como o token funciona por dentro |
| 4 | `OAuth2PasswordBearer` | Login e rota protegida |
| 5 | Autorização por papel | 401 ≠ 403 |
| 6 | Middleware | O que roda em toda requisição |
| 7 | CORS | O erro que todo front encontra |
| 8 | Filtros e ordenação | 🔴 Lista branca contra injeção |

> ⚠️ **Aviso honesto.** Esta aula ensina os mecanismos corretos, mas segurança de verdade é um campo inteiro. Antes de expor uma API ao público, leia o **OWASP API Security Top 10** e considere um provedor de identidade pronto (Auth0, Keycloak, Cognito) em vez de escrever o seu.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 06 · Aula 04
# ═══════════════════════════════════════════════════════════════
import json
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("sqlalchemy", "sqlalchemy"),
               ("pydantic-settings", "pydantic_settings"),
               ("pyjwt", "jwt"),
               ("python-multipart", "multipart"),
               ("bcrypt", "bcrypt")]:
    _garantir(_p, _m)

import fastapi
from fastapi.testclient import TestClient

print(f"✅ FastAPI {fastapi.__version__}")

# ═══════════════════════════════════════════════════════════════
#  Pasta de trabalho
# ═══════════════════════════════════════════════════════════════
BASE = Path("aula_06_04").resolve()
if BASE.exists():
    shutil.rmtree(BASE)
(BASE / "app" / "rotas").mkdir(parents=True)
sys.path = [str(BASE)] + [p for p in sys.path if p != str(BASE)]
print(f"📁 {BASE}")


# ═══════════════════════════════════════════════════════════════
#  Configuração do ambiente
#  Em produção estas variáveis viriam do orquestrador (Docker,
#  Kubernetes, systemd). Aqui simulamos isso no próprio processo.
# ═══════════════════════════════════════════════════════════════
import os
import secrets

os.environ["ATLAS_AMBIENTE"] = "desenvolvimento"
os.environ["ATLAS_CHAVE_SECRETA"] = secrets.token_urlsafe(48)
os.environ["ATLAS_MINUTOS_TOKEN"] = "30"
print(f"🔑 ATLAS_CHAVE_SECRETA definida ({len(os.environ['ATLAS_CHAVE_SECRETA'])} chars)")


# ═══════════════════════════════════════════════════════════════
#  Auxiliares
# ═══════════════════════════════════════════════════════════════

def req(cliente, metodo: str, caminho: str, mostrar_corpo=True, **kwargs):
    resposta = getattr(cliente, metodo.lower())(caminho, **kwargs)
    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    print(f"{cor} {metodo.upper():<7} {caminho:<40} → {resposta.status_code}")
    if kwargs.get("json") is not None:
        corpo = json.dumps(kwargs["json"], ensure_ascii=False)
        print(f"   envio  : {corpo[:120]}{'...' if len(corpo) > 120 else ''}")
    if mostrar_corpo:
        try:
            texto = json.dumps(resposta.json(), ensure_ascii=False, indent=2)
            linhas = texto.splitlines()
            for linha in linhas[:12]:
                print(f"   {linha}")
            if len(linhas) > 12:
                print(f"   ... (+{len(linhas) - 12} linhas)")
        except Exception:
            if resposta.text.strip():
                print(f"   {resposta.text[:200]}")
    print()
    return resposta


def arvore(raiz: Path, prefixo=""):
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name != "__pycache__"]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        print(f"{prefixo}{'└── ' if ultimo else '├── '}{item.name}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


def recarregar(*modulos):
    for nome in list(sys.modules):
        if any(nome == m or nome.startswith(m + ".") for m in modulos):
            del sys.modules[nome]


print("✅ Auxiliares prontos")

## 1. 🔴 Segredo não mora no código

O `config.py` da aula anterior tinha `os.getenv(...)` — está no caminho certo, mas sem validação e sem organização. O `pydantic-settings` resolve isso.

In [ ]:
%%writefile aula_06_04/.env.exemplo
# ═══════════════════════════════════════════════════════════
#  Modelo do arquivo de configuração.
#  Copie para `.env` e preencha com os valores reais.
#
#  🔴 `.env.exemplo` VAI para o Git. `.env` NUNCA vai.
# ═══════════════════════════════════════════════════════════
ATLAS_AMBIENTE=desenvolvimento
ATLAS_URL_BANCO=sqlite:///./atlas.db
ATLAS_CHAVE_SECRETA=troque-isto-por-64-bytes-aleatorios
ATLAS_MINUTOS_TOKEN=30
ATLAS_ORIGENS_PERMITIDAS=["http://localhost:3000"]

In [ ]:
%%writefile aula_06_04/.gitignore
# 🔴 O arquivo mais importante deste projeto
.env
*.db
*.db-wal
*.db-shm
__pycache__/

In [ ]:
%%writefile aula_06_04/app/config.py
"""Configuração validada, vinda do ambiente."""
from functools import lru_cache

from pydantic import Field, field_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class Config(BaseSettings):
    """Lê de variáveis de ambiente e do arquivo .env.

    Ordem de precedência (a primeira que existir vence):
      1. variável de ambiente do processo
      2. arquivo .env
      3. default declarado aqui
    """

    model_config = SettingsConfigDict(
        env_file=".env",
        env_prefix="ATLAS_",       # ATLAS_CHAVE_SECRETA → chave_secreta
        env_file_encoding="utf-8",
        extra="ignore",
    )

    ambiente: str = "desenvolvimento"
    url_banco: str = "sqlite:///./atlas.db"

    # 🔴 Sem default útil: se ninguém definir, o app não sobe. É de propósito.
    chave_secreta: str = Field(min_length=32)
    algoritmo: str = "HS256"
    minutos_token: int = Field(default=30, ge=1, le=1440)

    origens_permitidas: list[str] = ["http://localhost:3000"]

    @field_validator("chave_secreta")
    @classmethod
    def chave_nao_pode_ser_o_exemplo(cls, v: str) -> str:
        proibidas = {"segredo", "changeme", "troque-isto-por-64-bytes-aleatorios"}
        if v.lower() in proibidas:
            raise ValueError("a chave secreta ainda é a de exemplo")
        return v

    @property
    def producao(self) -> bool:
        return self.ambiente == "producao"


@lru_cache
def obter_config() -> Config:
    """🔑 lru_cache: lê o ambiente UMA vez por processo.

    Também é o ponto de substituição nos testes:
        obter_config.cache_clear()
    """
    return Config()

In [ ]:
# A configuração falha ALTO quando está errada
recarregar("app")
from app.config import Config  # noqa: E402

# Escondemos a variável só para ver o app recusar-se a subir
_guardada = os.environ.pop("ATLAS_CHAVE_SECRETA")

print("── sem chave secreta ──")
try:
    Config(_env_file=None)
except Exception as erro:
    print(f"   🔴 {erro.errors()[0]['loc'][0]}: {erro.errors()[0]['msg']}")

print("\n── chave curta demais ──")
try:
    Config(_env_file=None, chave_secreta="123")
except Exception as erro:
    print(f"   🔴 {erro.errors()[0]['msg']}")

print("\n── com a chave de exemplo ──")
try:
    Config(_env_file=None, chave_secreta="troque-isto-por-64-bytes-aleatorios")
except Exception as erro:
    print(f"   🔴 {erro.errors()[0]['msg']}")

os.environ["ATLAS_CHAVE_SECRETA"] = _guardada       # devolve

print("\n── configuração válida ──")
config = Config(_env_file=None)
print(f"   ambiente : {config.ambiente}")
print(f"   token    : {config.minutos_token} min")
print(f"   chave    : {config.chave_secreta[:12]}… ({len(config.chave_secreta)} chars)")
print(f"   produção : {config.producao}")

> 🎯 **Falhar ao subir é melhor do que funcionar errado.**
>
> Sem `Field(min_length=32)`, uma chave secreta vazia deixaria o app subir e assinar tokens que qualquer um forja. Com ele, o container morre no `docker run` e você descobre na hora — não três semanas depois.
>
> 🔴 **Como gerar uma chave de verdade:**
> ```bash
> python -c "import secrets; print(secrets.token_urlsafe(48))"
> ```
> Nunca `"segredo"`, nunca a mesma em dev e produção, nunca versionada.
>
> 💭 **E se vazar?** Trocar a chave invalida **todos** os tokens emitidos. É por isso que tokens são curtos e existe *refresh token* — assunto para depois.

## 2. 🔴 Senha: você nunca guarda, você guarda o hash

In [ ]:
%%writefile aula_06_04/app/seguranca.py
"""Hash de senha e emissão/verificação de token.

🔴 REGRA ABSOLUTA: a senha em texto puro nunca é gravada, nunca é logada,
   nunca aparece numa resposta. Só existe na memória, por milissegundos.
"""
from datetime import datetime, timedelta, timezone

import bcrypt
import jwt

from app.config import obter_config


# ═══════════════════════════════════════════════════════════════
#  Senha
# ═══════════════════════════════════════════════════════════════
def gerar_hash(senha: str) -> str:
    """bcrypt: lento DE PROPÓSITO, e com sal embutido.

    Cada chamada gera um sal aleatório novo — por isso a mesma senha
    produz hashes diferentes. Isso derrota tabelas arco-íris.
    """
    return bcrypt.hashpw(senha.encode("utf-8"), bcrypt.gensalt(rounds=12)).decode()


def conferir_senha(senha: str, hash_guardado: str) -> bool:
    """Comparação em tempo constante — não vaza informação pelo relógio."""
    try:
        return bcrypt.checkpw(senha.encode("utf-8"), hash_guardado.encode("utf-8"))
    except ValueError:
        return False


# ═══════════════════════════════════════════════════════════════
#  Token (JWT)
# ═══════════════════════════════════════════════════════════════
def criar_token(assunto: str, papel: str, minutos: int | None = None) -> str:
    config = obter_config()
    expira_em = datetime.now(timezone.utc) + timedelta(
        minutes=minutos if minutos is not None else config.minutos_token)
    conteudo = {
        "sub": assunto,                       # quem é (claim padrão)
        "papel": papel,                       # claim nosso
        "exp": expira_em,                     # quando expira (padrão)
        "iat": datetime.now(timezone.utc),    # quando foi emitido
        "iss": "atlas-api",                   # quem emitiu
    }
    return jwt.encode(conteudo, config.chave_secreta, algorithm=config.algoritmo)


class TokenInvalido(Exception):
    pass


def ler_token(token: str) -> dict:
    """Valida assinatura E expiração. As duas coisas."""
    config = obter_config()
    try:
        return jwt.decode(token, config.chave_secreta,
                          algorithms=[config.algoritmo], issuer="atlas-api")
    except jwt.ExpiredSignatureError as erro:
        raise TokenInvalido("token expirado") from erro
    except jwt.InvalidTokenError as erro:
        raise TokenInvalido("token inválido") from erro

In [ ]:
recarregar("app")
from app.seguranca import conferir_senha, gerar_hash  # noqa: E402

senha = "aurora-2026"
h1 = gerar_hash(senha)
h2 = gerar_hash(senha)

print(f"hash 1: {h1}")
print(f"hash 2: {h2}")
print(f"\nIguais? {h1 == h2}   ← 🎯 sal aleatório diferente a cada vez\n")

print(f"confere senha certa  : {conferir_senha(senha, h1)}")
print(f"confere senha errada : {conferir_senha('aurora-2025', h1)}")
print(f"confere no outro hash: {conferir_senha(senha, h2)}   ← ambos válidos")

# Anatomia do hash bcrypt
partes = h1.split("$")
print(f"\nalgoritmo={partes[1]}  custo=2^{partes[2]}  sal+hash={partes[3][:12]}…")

In [ ]:
# 🎯 Por que bcrypt e não sha256?
import hashlib
import time

def medir(f, n=3):
    inicio = time.perf_counter()
    for _ in range(n):
        f()
    return (time.perf_counter() - inicio) / n * 1000

ms_sha = medir(lambda: hashlib.sha256(b"aurora-2026").hexdigest(), n=1000)
ms_bcrypt = medir(lambda: gerar_hash("aurora-2026"))

print(f"sha256 : {ms_sha:9.4f} ms")
print(f"bcrypt : {ms_bcrypt:9.4f} ms")
print(f"\nbcrypt é ~{ms_bcrypt / ms_sha:,.0f}× mais lento — e isso é a FUNCIONALIDADE.")
print("\nNuma lista vazada, testar 1 bilhão de senhas em UM núcleo levaria:")
print(f"   com sha256 : {1e9 * ms_sha / 1000 / 3600:>10,.1f} horas")
print(f"   com bcrypt : {1e9 * ms_bcrypt / 1000 / 3600 / 24 / 365:>10,.0f} anos")
print("\n⚠️ Números ilustrativos: um atacante real usa GPUs e milhares de")
print("   núcleos. A conclusão não muda — muda a escala dos dois lados.")

> 🔴 **`sha256(senha)` não é hash de senha.** SHA-256 foi feito para ser **rápido** — é ótimo para verificar integridade de arquivo e péssimo para senha, porque a velocidade que ajuda você ajuda mais ainda quem tem a lista vazada e uma GPU.
>
> **Use bcrypt, scrypt ou Argon2.** Todos são lentos de propósito e têm custo ajustável: quando o hardware melhorar, você aumenta o `rounds` e continua na frente.
>
> ⚠️ **O bcrypt trunca em 72 bytes.** Senhas mais longas têm o excedente ignorado silenciosamente. Se você permite senhas longas, valide o tamanho ou use Argon2.

## 3. Dentro do JWT

Um JWT não é criptografia. É um texto **assinado** — qualquer um lê, ninguém altera sem a chave.

In [ ]:
import base64

from app.seguranca import TokenInvalido, criar_token, ler_token

token = criar_token("ana@aurora.com.br", "admin")
cabecalho, carga, assinatura = token.split(".")

print(f"token ({len(token)} chars):\n   {token[:60]}…\n")


def decodificar(parte: str) -> dict:
    """🔴 Sem chave nenhuma. É só base64."""
    return json.loads(base64.urlsafe_b64decode(parte + "=" * (-len(parte) % 4)))


print(f"cabeçalho  : {decodificar(cabecalho)}")
print(f"carga      : {decodificar(carga)}")
print(f"assinatura : {assinatura[:24]}…  (só quem tem a chave produz)")

> 🔴 **Leia de novo: eu li a carga sem chave nenhuma.**
>
> JWT é **assinado**, não **criptografado**. Qualquer um com o token vê o conteúdo — basta colar em jwt.io.
>
> **Portanto, nunca coloque num JWT:** CPF, endereço, telefone, saldo, senha, chave de API. Coloque um identificador e busque o resto no banco.
>
> 💭 **O que a assinatura garante:** que o conteúdo **não foi alterado** e que veio de quem tem a chave. Só isso — e é bastante.

In [ ]:
# Prova: alterar a carga invalida o token
carga_falsa = decodificar(carga) | {"papel": "admin", "sub": "invasor@fora.com"}
carga_falsa.pop("exp", None)
nova = base64.urlsafe_b64encode(json.dumps(carga_falsa).encode()).decode().rstrip("=")
token_forjado = f"{cabecalho}.{nova}.{assinatura}"

for nome, t in [("token legítimo", token), ("token forjado", token_forjado)]:
    try:
        dados = ler_token(t)
        print(f"✅ {nome:<16} → sub={dados['sub']} papel={dados['papel']}")
    except TokenInvalido as erro:
        print(f"🔴 {nome:<16} → recusado: {erro}")

In [ ]:
# Expiração: o token morre sozinho
import time as _t

curto = criar_token("bruno@aurora.com.br", "leitor", minutos=0)
_t.sleep(1.1)
try:
    ler_token(curto)
except TokenInvalido as erro:
    print(f"🔴 {erro}")

print("\n💡 Token curto limita o estrago de um vazamento.")
print("   Padrão comum: acesso de 15–30 min + refresh token de dias.")

## 4. Login e rota protegida

In [ ]:
%%writefile aula_06_04/app/usuarios.py
"""Repositório de usuários — em memória, para a aula.

Num projeto real isto vira uma tabela com SQLAlchemy, exatamente como
`produtos` na aula anterior.
"""
from app.seguranca import gerar_hash

# 🔴 Repare: guardamos `senha_hash`, nunca `senha`.
USUARIOS: dict[str, dict] = {
    "ana@aurora.com.br": {
        "email": "ana@aurora.com.br", "nome": "Ana Prado",
        "papel": "admin", "ativo": True,
        "senha_hash": gerar_hash("aurora-admin-2026"),
    },
    "bruno@aurora.com.br": {
        "email": "bruno@aurora.com.br", "nome": "Bruno Lima",
        "papel": "operador", "ativo": True,
        "senha_hash": gerar_hash("aurora-op-2026"),
    },
    "carla@aurora.com.br": {
        "email": "carla@aurora.com.br", "nome": "Carla Dias",
        "papel": "leitor", "ativo": True,
        "senha_hash": gerar_hash("aurora-leitor-2026"),
    },
    "davi@aurora.com.br": {
        "email": "davi@aurora.com.br", "nome": "Davi Souza",
        "papel": "operador", "ativo": False,          # 🔒 desligado
        "senha_hash": gerar_hash("aurora-ex-2026"),
    },
}


def buscar(email: str) -> dict | None:
    return USUARIOS.get(email.lower().strip())

In [ ]:
%%writefile aula_06_04/app/dependencias.py
"""Dependências de segurança."""
from typing import Annotated

from fastapi import Depends, HTTPException, Query, status
from fastapi.security import OAuth2PasswordBearer

from app import usuarios
from app.seguranca import TokenInvalido, ler_token

# 🔑 tokenUrl é o endereço do login. Serve para o /docs mostrar o botão
#    "Authorize" — o esquema vira parte da documentação OpenAPI.
esquema_oauth = OAuth2PasswordBearer(tokenUrl="/auth/token")

# 🔴 O 401 SEMPRE acompanha WWW-Authenticate. É o que a especificação manda
#    e o que faz o cliente saber como se autenticar.
_CABECALHO = {"WWW-Authenticate": "Bearer"}


def usuario_atual(token: Annotated[str, Depends(esquema_oauth)]) -> dict:
    """Traduz token → usuário. Levanta 401 em qualquer problema."""
    try:
        dados = ler_token(token)
    except TokenInvalido as erro:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, str(erro),
                            headers=_CABECALHO) from erro

    usuario = usuarios.buscar(dados.get("sub", ""))
    if usuario is None:
        # 🔒 Token assinado por nós, mas o usuário sumiu do banco.
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "usuário não existe",
                            headers=_CABECALHO)
    if not usuario["ativo"]:
        raise HTTPException(status.HTTP_403_FORBIDDEN, "usuário desativado")
    return usuario


UsuarioDep = Annotated[dict, Depends(usuario_atual)]


# ═══════════════════════════════════════════════════════════════
#  Autorização por papel — uma FÁBRICA de dependências
# ═══════════════════════════════════════════════════════════════
HIERARQUIA = {"leitor": 0, "operador": 1, "admin": 2}


def exigir_papel(minimo: str):
    """Devolve uma dependência que exige pelo menos `minimo`.

    💡 O padrão é o mesmo dos decoradores parametrizados do M04: uma
       função que fabrica outra função, fechando sobre `minimo`.
    """
    def verificar(usuario: UsuarioDep) -> dict:
        if HIERARQUIA[usuario["papel"]] < HIERARQUIA[minimo]:
            raise HTTPException(
                status.HTTP_403_FORBIDDEN,
                f"requer papel '{minimo}'; você é '{usuario['papel']}'")
        return usuario
    return verificar


OperadorDep = Annotated[dict, Depends(exigir_papel("operador"))]
AdminDep = Annotated[dict, Depends(exigir_papel("admin"))]


# ═══════════════════════════════════════════════════════════════
#  Filtros e ordenação
# ═══════════════════════════════════════════════════════════════
# 🔴 LISTA BRANCA. Nome de coluna não é parametrizável com `?` —
#    se veio do usuário e vira string SQL, tem que ser validado aqui.
CAMPOS_ORDENAVEIS = ("sku", "nome", "preco", "estoque", "categoria")


def ordenacao(
    ordenar_por: Annotated[str, Query(description="Campo de ordenação")] = "sku",
    direcao: Annotated[str, Query(pattern="^(asc|desc)$")] = "asc",
) -> dict:
    if ordenar_por not in CAMPOS_ORDENAVEIS:
        raise HTTPException(
            status.HTTP_422_UNPROCESSABLE_ENTITY,
            f"ordenar_por deve ser um de {list(CAMPOS_ORDENAVEIS)}")
    return {"campo": ordenar_por, "desc": direcao == "desc"}


def paginacao(
    pagina: Annotated[int, Query(ge=1)] = 1,
    por_pagina: Annotated[int, Query(ge=1, le=100)] = 20,
) -> dict:
    return {"pular": (pagina - 1) * por_pagina, "limite": por_pagina,
            "pagina": pagina, "por_pagina": por_pagina}


OrdenacaoDep = Annotated[dict, Depends(ordenacao)]
PaginacaoDep = Annotated[dict, Depends(paginacao)]

> 🔴 **Por que a lista branca em `ordenacao` não é opcional.**
>
> `ORDER BY` não aceita placeholder. Se você escrever `f"ORDER BY {campo}"` com o valor cru da query string, acabou de dar ao mundo uma injeção de SQL.
>
> Mesmo usando SQLAlchemy com `getattr(Produto, campo)`, um campo arbitrário permite ordenar por colunas internas e **inferir dados** — ordenar por `custo` revela a margem sem nunca lê-la.
>
> 🧭 **A regra:** se um valor do usuário vira **identificador** (coluna, tabela, nome de arquivo, caminho), valide contra uma lista fechada. Placeholders só protegem **valores**.

In [ ]:
%%writefile aula_06_04/app/rotas/auth.py
"""Login."""
from typing import Annotated

from fastapi import APIRouter, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordRequestForm
from pydantic import BaseModel

from app import usuarios
from app.config import obter_config
from app.dependencias import UsuarioDep
from app.seguranca import conferir_senha, criar_token

roteador = APIRouter(prefix="/auth", tags=["Autenticação"])


class Token(BaseModel):
    access_token: str
    token_type: str = "bearer"
    expira_em_minutos: int


class UsuarioResposta(BaseModel):
    email: str
    nome: str
    papel: str
    # 🔒 `senha_hash` não está aqui. Nunca.


@roteador.post("/token", response_model=Token)
def entrar(formulario: Annotated[OAuth2PasswordRequestForm, Depends()]):
    """Login. O OAuth2 manda usar `username`/`password` em form-data.

    🔴 A mensagem de erro é a MESMA para e-mail inexistente e senha
       errada. Dizer "e-mail não cadastrado" entrega ao atacante a
       lista de quem tem conta.
    """
    generico = HTTPException(status.HTTP_401_UNAUTHORIZED,
                             "e-mail ou senha incorretos",
                             headers={"WWW-Authenticate": "Bearer"})

    usuario = usuarios.buscar(formulario.username)
    if usuario is None or not conferir_senha(formulario.password, usuario["senha_hash"]):
        raise generico
    if not usuario["ativo"]:
        raise HTTPException(status.HTTP_403_FORBIDDEN, "usuário desativado")

    config = obter_config()
    return Token(access_token=criar_token(usuario["email"], usuario["papel"]),
                 expira_em_minutos=config.minutos_token)


@roteador.get("/eu", response_model=UsuarioResposta)
def quem_sou_eu(usuario: UsuarioDep):
    return usuario

> ⚠️ **`Annotated[OAuth2PasswordRequestForm, Depends()]`** — `Depends()` vazio significa *"use a própria classe anotada como dependência"*. O FastAPI instancia `OAuth2PasswordRequestForm` a partir do `form-data`.
>
> 🔴 **Por que `form-data` e não JSON?** Porque a especificação OAuth2 manda. É chato, mas é o que faz o botão **Authorize** do `/docs` funcionar e o que todo cliente OAuth espera.
>
> ⚠️ Isso exige o pacote **`python-multipart`** instalado.

## 5. Middleware — o que roda em toda requisição

In [ ]:
%%writefile aula_06_04/app/main.py
"""Aplicação completa, com segurança."""
import logging
import time
import uuid

from fastapi import FastAPI, Request, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

from app.config import obter_config
from app.rotas import auth, produtos

logging.basicConfig(level=logging.INFO, format="   [log] %(message)s", force=True)
# 💡 Bibliotecas de terceiros são faladeiras demais em INFO.
logging.getLogger("httpx").setLevel(logging.WARNING)
registro = logging.getLogger("atlas")

config = obter_config()
app = FastAPI(title="Atlas API", version="3.0.0")


# ═══════════════════════════════════════════════════════════════
#  CORS — sempre o middleware mais EXTERNO
# ═══════════════════════════════════════════════════════════════
app.add_middleware(
    CORSMiddleware,
    # 🔴 Nunca ["*"] junto com allow_credentials=True. Os navegadores
    #    recusam essa combinação — e com razão: seria abrir a API
    #    autenticada para qualquer site.
    allow_origins=config.origens_permitidas,
    allow_credentials=True,
    allow_methods=["GET", "POST", "PATCH", "DELETE"],
    allow_headers=["Authorization", "Content-Type"],
    max_age=600,
)


# ═══════════════════════════════════════════════════════════════
#  Middleware próprio: id de correlação + tempo
# ═══════════════════════════════════════════════════════════════
@app.middleware("http")
async def rastrear(requisicao: Request, proximo):
    """Roda ANTES e DEPOIS de cada requisição.

    ⚠️ Middleware é `async` e roda para TODAS as rotas, inclusive as que
       nem existem. Nada de I/O bloqueante aqui — você atrasaria tudo.
    """
    id_req = requisicao.headers.get("X-Request-ID") or uuid.uuid4().hex[:12]
    inicio = time.perf_counter()

    resposta = await proximo(requisicao)          # ← a rota acontece aqui

    ms = (time.perf_counter() - inicio) * 1000
    resposta.headers["X-Request-ID"] = id_req
    resposta.headers["X-Tempo-ms"] = f"{ms:.1f}"
    registro.info("%s %s → %s em %.1fms [%s]", requisicao.method,
                  requisicao.url.path, resposta.status_code, ms, id_req)
    return resposta


# ═══════════════════════════════════════════════════════════════
#  Cabeçalhos de segurança
# ═══════════════════════════════════════════════════════════════
@app.middleware("http")
async def cabecalhos_seguros(requisicao: Request, proximo):
    resposta = await proximo(requisicao)
    resposta.headers["X-Content-Type-Options"] = "nosniff"
    resposta.headers["X-Frame-Options"] = "DENY"
    resposta.headers["Referrer-Policy"] = "no-referrer"
    if obter_config().producao:
        resposta.headers["Strict-Transport-Security"] = "max-age=31536000"
    return resposta


# ═══════════════════════════════════════════════════════════════
#  Erro inesperado: log detalhado, resposta genérica
# ═══════════════════════════════════════════════════════════════
@app.exception_handler(Exception)
def erro_inesperado(requisicao: Request, erro: Exception):
    registro.error("erro não tratado em %s: %s", requisicao.url.path, erro)
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={"codigo": "erro_interno", "mensagem": "Erro interno."})


app.include_router(auth.roteador)
app.include_router(produtos.roteador)


@app.get("/saude", tags=["Infra"])
def saude():
    return {"status": "ok", "ambiente": config.ambiente}

> ⚠️ **A ordem dos middlewares é ao contrário do que parece.**
>
> O último `add_middleware` fica **mais perto da rota**; o primeiro fica **mais externo**. Um `@app.middleware("http")` é açúcar para `add_middleware`, então os decoradores também entram nessa pilha.
>
> **Consequência prática:** o CORS precisa ser o mais externo, senão uma resposta de erro gerada por outro middleware sai sem os cabeçalhos CORS — e o navegador esconde o erro real do front-end atrás de "CORS policy".

## 6. Rotas protegidas

In [ ]:
%%writefile aula_06_04/app/rotas/produtos.py
"""Catálogo — cada rota com o nível de acesso que faz sentido."""
from fastapi import APIRouter, HTTPException, Query, status
from pydantic import BaseModel, Field
from typing import Annotated

from app.dependencias import (AdminDep, OperadorDep, OrdenacaoDep,
                              PaginacaoDep, UsuarioDep)

roteador = APIRouter(prefix="/produtos", tags=["Produtos"])

CATALOGO: dict[str, dict] = {
    "NB-DELL-15": {"sku": "NB-DELL-15", "nome": "Notebook Dell Inspiron 15",
                   "categoria": "Notebooks", "preco": 2599.90, "custo": 2120.00,
                   "estoque": 14},
    "MO-LG-24UW": {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide",
                   "categoria": "Monitores", "preco": 1199.00, "custo": 920.00,
                   "estoque": 31},
    "PE-LOG-M170": {"sku": "PE-LOG-M170", "nome": "Mouse Logitech M170",
                    "categoria": "Periféricos", "preco": 89.90, "custo": 52.00,
                    "estoque": 120},
    "PE-RED-K552": {"sku": "PE-RED-K552", "nome": "Teclado Redragon K552",
                    "categoria": "Periféricos", "preco": 249.90, "custo": 168.00,
                    "estoque": 8},
    "AR-KING-1TB": {"sku": "AR-KING-1TB", "nome": "SSD Kingston NV2 1TB",
                    "categoria": "Armazenamento", "preco": 429.00, "custo": 305.00,
                    "estoque": 0},
}


class ProdutoResposta(BaseModel):
    sku: str
    nome: str
    categoria: str
    preco: float
    estoque: int
    # 🔒 sem `custo`


class ProdutoAdmin(ProdutoResposta):
    """Só quem é admin enxerga o custo e a margem."""
    custo: float
    margem_pct: float


class Pagina(BaseModel):
    total: int
    pagina: int
    por_pagina: int
    itens: list[ProdutoResposta]


class AjusteEstoque(BaseModel):
    delta: int = Field(description="Positivo entra, negativo sai")


# ── 🔓 Qualquer usuário autenticado ──
@roteador.get("", response_model=Pagina)
def listar(usuario: UsuarioDep, pag: PaginacaoDep, ord_: OrdenacaoDep,
           categoria: str | None = None,
           preco_min: Annotated[float | None, Query(ge=0)] = None,
           preco_max: Annotated[float | None, Query(ge=0)] = None,
           busca: Annotated[str | None, Query(min_length=2, max_length=60)] = None,
           somente_disponiveis: bool = False):
    itens = list(CATALOGO.values())

    if categoria:
        itens = [p for p in itens if p["categoria"].lower() == categoria.lower()]
    if preco_min is not None:
        itens = [p for p in itens if p["preco"] >= preco_min]
    if preco_max is not None:
        itens = [p for p in itens if p["preco"] <= preco_max]
    if busca:
        alvo = busca.lower()
        itens = [p for p in itens if alvo in p["nome"].lower()]
    if somente_disponiveis:
        itens = [p for p in itens if p["estoque"] > 0]

    # 🔒 `ord_["campo"]` já passou pela lista branca da dependência
    itens.sort(key=lambda p: p[ord_["campo"]], reverse=ord_["desc"])

    inicio = pag["pular"]
    return {"total": len(itens), "pagina": pag["pagina"],
            "por_pagina": pag["por_pagina"],
            "itens": itens[inicio:inicio + pag["limite"]]}


@roteador.get("/{sku}", response_model=ProdutoResposta)
def obter(sku: str, usuario: UsuarioDep):
    if sku not in CATALOGO:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "produto não encontrado")
    return CATALOGO[sku]


# ── 🔐 Operador ou acima ──
@roteador.patch("/{sku}/estoque", response_model=ProdutoResposta)
def ajustar_estoque(sku: str, ajuste: AjusteEstoque, operador: OperadorDep):
    if sku not in CATALOGO:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "produto não encontrado")
    novo = CATALOGO[sku]["estoque"] + ajuste.delta
    if novo < 0:
        raise HTTPException(status.HTTP_409_CONFLICT,
                            f"estoque ficaria {novo}; disponível {CATALOGO[sku]['estoque']}")
    CATALOGO[sku]["estoque"] = novo
    return CATALOGO[sku]


# ── 🔒 Só admin ──
@roteador.get("/{sku}/interno", response_model=ProdutoAdmin)
def ver_interno(sku: str, admin: AdminDep):
    """O MESMO produto, com os campos que só a diretoria vê."""
    if sku not in CATALOGO:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "produto não encontrado")
    p = CATALOGO[sku]
    margem = (p["preco"] - p["custo"]) / p["preco"] * 100 if p["preco"] else 0
    return {**p, "margem_pct": round(margem, 1)}


@roteador.delete("/{sku}", status_code=status.HTTP_204_NO_CONTENT)
def remover(sku: str, admin: AdminDep):
    if sku not in CATALOGO:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "produto não encontrado")
    del CATALOGO[sku]

In [ ]:
%%writefile aula_06_04/app/__init__.py
"""Atlas API — versão com segurança."""

In [ ]:
%%writefile aula_06_04/app/rotas/__init__.py
"""Routers."""

In [ ]:
print("Estrutura:\n")
arvore(BASE)

## 7. A API em uso

In [ ]:
recarregar("app")
from app.main import app  # noqa: E402

cliente = TestClient(app)

print("── sem token ──")
req(cliente, "GET", "/produtos")

print("── token inventado ──")
req(cliente, "GET", "/produtos", headers={"Authorization": "Bearer abc.def.ghi"})

In [ ]:
# Login
def entrar(email: str, senha: str) -> dict:
    """Devolve o cabeçalho pronto — ou mostra o erro."""
    r = cliente.post("/auth/token", data={"username": email, "password": senha})
    if r.status_code != 200:
        print(f"   🔴 {email:<22} {r.status_code} {r.json()['detail']}")
        return {}
    print(f"   ✅ {email:<22} token de {r.json()['expira_em_minutos']} min")
    return {"Authorization": f"Bearer {r.json()['access_token']}"}


print("── logins ──")
ana = entrar("ana@aurora.com.br", "aurora-admin-2026")        # admin
bruno = entrar("bruno@aurora.com.br", "aurora-op-2026")       # operador
carla = entrar("carla@aurora.com.br", "aurora-leitor-2026")   # leitor
entrar("ana@aurora.com.br", "senha-errada")
entrar("ninguem@aurora.com.br", "qualquer")
entrar("davi@aurora.com.br", "aurora-ex-2026")                # desativado

> 🔴 **Compare as duas últimas linhas.**
>
> `ana@aurora.com.br` com senha errada e `ninguem@aurora.com.br` recebem **a mesma resposta**. Um atacante não consegue descobrir quem tem conta testando e-mails.
>
> 💭 Rigorosamente, ainda dá para inferir pelo **tempo**: quando o e-mail não existe, pulamos o bcrypt e a resposta volta mais rápido. A defesa é rodar um hash falso mesmo quando o usuário não existe. Fica como exercício.

In [ ]:
print("── /auth/eu ──")
req(cliente, "GET", "/auth/eu", headers=ana)
req(cliente, "GET", "/auth/eu", headers=carla)

In [ ]:
# 🎯 A MESMA rota, três papéis
print("═══ PATCH /produtos/PE-RED-K552/estoque (exige operador) ═══")
for nome, cab in [("carla (leitor)  ", carla), ("bruno (operador)", bruno),
                  ("ana (admin)     ", ana)]:
    r = cliente.patch("/produtos/PE-RED-K552/estoque", json={"delta": 5}, headers=cab)
    marca = "✅" if r.status_code == 200 else "🔴"
    detalhe = (f"estoque agora {r.json()['estoque']}" if r.status_code == 200
               else r.json()["detail"])
    print(f"   {marca} {nome} → {r.status_code}  {detalhe}")

In [ ]:
print("═══ GET /produtos/NB-DELL-15/interno (só admin) ═══")
for nome, cab in [("bruno (operador)", bruno), ("ana (admin)     ", ana)]:
    r = cliente.get("/produtos/NB-DELL-15/interno", headers=cab)
    marca = "✅" if r.status_code == 200 else "🔴"
    print(f"   {marca} {nome} → {r.status_code}  {r.json()}")

print("\n💡 Mesmo produto, dois contratos de saída. O `custo` só aparece")
print("   na rota que exige admin — e por causa do `response_model`.")

In [ ]:
# Regra de negócio ainda vale, mesmo com permissão
print("── operador tentando negativar o estoque ──")
req(cliente, "PATCH", "/produtos/PE-RED-K552/estoque",
    json={"delta": -999}, headers=bruno)

print("── admin removendo ──")
req(cliente, "DELETE", "/produtos/AR-KING-1TB", headers=ana, mostrar_corpo=False)
req(cliente, "GET", "/produtos/AR-KING-1TB", headers=ana)

> 🎯 **401 vs 403 — a distinção que quase todo mundo erra:**
>
> | Código | Significado | Exemplo aqui |
> |--------|-------------|--------------|
> | **401** Unauthorized | *"Não sei quem você é"* | sem token, token expirado, forjado |
> | **403** Forbidden | *"Sei quem você é, e não pode"* | Carla tentando ajustar estoque |
>
> O nome do 401 é infeliz — deveria ser *Unauthenticated*. O **401 pede que você se identifique**; o **403 diz que identificar-se não adianta**.

## 8. Filtros, ordenação e paginação

In [ ]:
def listar(rotulo, **params):
    r = cliente.get("/produtos", params=params, headers=carla)
    if r.status_code != 200:
        print(f"🔴 {rotulo}: {r.status_code} {r.json()['detail']}")
        return
    dados = r.json()
    print(f"▸ {rotulo}  (total {dados['total']}, página {dados['pagina']})")
    for p in dados["itens"]:
        print(f"     {p['sku']:<13} {p['nome'][:26]:<26} R$ {p['preco']:>8.2f}  "
              f"est {p['estoque']:>3}")
    print()


listar("tudo")
listar("Periféricos", categoria="Periféricos")
listar("acima de R$ 400", preco_min=400)
listar("busca 'monitor'", busca="monitor")
listar("mais caros primeiro", ordenar_por="preco", direcao="desc")
listar("página 2, 2 por página", por_pagina=2, pagina=2)

In [ ]:
# 🔴 A lista branca em ação
print("── tentando ordenar por um campo que não é público ──")
listar("por custo", ordenar_por="custo")

print("── tentando injetar SQL pelo nome do campo ──")
listar("injeção", ordenar_por="preco; DROP TABLE produtos--")

print("── direção inválida (o `pattern` do Query pega) ──")
r = cliente.get("/produtos", params={"direcao": "aleatoria"}, headers=carla)
print(f"🔴 {r.status_code} {r.json()['detail'][0]['msg']}")

> 🔴 **Repare no que aconteceu com `ordenar_por=custo`.**
>
> `custo` é um campo **real** do dicionário — `itens.sort(key=lambda p: p["custo"])` funcionaria perfeitamente. E ordenar por custo, mesmo sem exibi-lo, **revela a ordem de margem de todo o catálogo**.
>
> A lista branca não bloqueou uma injeção aqui. Bloqueou um **vazamento por canal lateral**. São dois motivos diferentes para a mesma defesa.

## 9. CORS na prática

In [ ]:
print("── requisição do front autorizado (localhost:3000) ──")
r = cliente.get("/saude", headers={"Origin": "http://localhost:3000"})
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"   {k}: {v}")

print("\n── requisição de origem NÃO autorizada ──")
r = cliente.get("/saude", headers={"Origin": "http://site-malicioso.com"})
print(f"   status {r.status_code}  ← o servidor RESPONDEU normalmente")
print(f"   corpo: {r.json()}")
print(f"   access-control-allow-origin: {r.headers.get('access-control-allow-origin', '🔴 AUSENTE')}")
print("\n   💡 É a AUSÊNCIA desse cabeçalho que faz o navegador jogar a")
print("      resposta fora e mostrar o erro de CORS ao front-end.")

In [ ]:
print("── preflight (o OPTIONS que o navegador manda sozinho) ──")
r = cliente.options("/produtos/NB-DELL-15", headers={
    "Origin": "http://localhost:3000",
    "Access-Control-Request-Method": "DELETE",
    "Access-Control-Request-Headers": "Authorization",
})
print(f"   status {r.status_code}")
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"   {k}: {v}")

> ⚠️ **CORS não é segurança do servidor.** Ele instrui o **navegador** a bloquear. `curl`, Postman e qualquer script Python ignoram CORS completamente — como o `TestClient` acabou de fazer.
>
> **Quem protege a API é a autenticação.** O CORS protege o *usuário* de um site malicioso usar as credenciais dele contra a sua API.
>
> 🔴 **`allow_origins=["*"]` com `allow_credentials=True`** é recusado pelos navegadores. Liste as origens de verdade.

In [ ]:
# Cabeçalhos que os middlewares acrescentaram
r = cliente.get("/saude")
print("Cabeçalhos da resposta:\n")
for k in ["x-request-id", "x-tempo-ms", "x-content-type-options",
          "x-frame-options", "referrer-policy"]:
    print(f"   {k:<26} {r.headers.get(k, '—')}")

print("\n── id de correlação enviado pelo cliente é PRESERVADO ──")
r = cliente.get("/saude", headers={"X-Request-ID": "trace-do-front-001"})
print(f"   x-request-id: {r.headers['x-request-id']}")

## 🔧 Prática guiada — auditoria da API

In [ ]:
espec = cliente.get("/openapi.json").json()

# Rotas que exigem autenticação, segundo o próprio OpenAPI
print(f"{'MÉTODO':<8} {'ROTA':<32} SEGURANÇA")
print("─" * 62)
protegidas = publicas = 0
for caminho, metodos in sorted(espec["paths"].items()):
    for metodo, detalhe in metodos.items():
        tem = "security" in detalhe
        protegidas += tem
        publicas += not tem
        print(f"{metodo.upper():<8} {caminho:<32} {'🔒 token' if tem else '🔓 pública'}")

print(f"\n{protegidas} protegida(s), {publicas} pública(s)")
print(f"Esquemas de segurança: {list(espec['components'].get('securitySchemes', {}))}")

In [ ]:
# 🔒 Auditoria: nenhum esquema de SAÍDA pode conter campo sensível
SENSIVEIS = {"custo", "senha", "senha_hash", "password", "cpf", "token_interno"}
EXCECOES = {"ProdutoAdmin"}          # intencional: a rota exige admin

# 🎯 Só interessam os esquemas usados em RESPOSTAS. Um esquema de
#    entrada com `password` é normal — é o formulário de login.
de_saida: set[str] = set()
for metodos in espec["paths"].values():
    for detalhe in metodos.values():
        for resposta in detalhe.get("responses", {}).values():
            for tipo in resposta.get("content", {}).values():
                ref = tipo.get("schema", {}).get("$ref", "")
                if ref:
                    de_saida.add(ref.rsplit("/", 1)[-1])

print(f"Esquemas de resposta encontrados: {len(de_saida)}\n")
for nome in sorted(de_saida):
    campos = set(espec["components"]["schemas"][nome].get("properties", {}))
    vazou = campos & SENSIVEIS
    if vazou and nome not in EXCECOES:
        print(f"   🔴 {nome:<22} expõe {sorted(vazou)}")
    elif vazou:
        print(f"   ⚠️  {nome:<22} expõe {sorted(vazou)} (intencional, exige admin)")
    else:
        print(f"   ✅ {nome:<22} limpo")

fora = sorted(set(espec["components"]["schemas"]) - de_saida)
print(f"\nNão auditados (entrada/erro): {fora}")

In [ ]:
# Checagem final: toda rota de escrita exige pelo menos operador
print("Rotas de escrita e quem consegue chamá-las:\n")
ESCRITA = [("PATCH", "/produtos/PE-LOG-M170/estoque", {"delta": 1}),
           ("DELETE", "/produtos/PE-LOG-M170", None)]

for metodo, caminho, corpo in ESCRITA:
    linha = f"   {metodo:<7} {caminho:<32}"
    for rotulo, cab in [("anônimo", {}), ("leitor", carla),
                        ("operador", bruno), ("admin", ana)]:
        kwargs = {"headers": cab}
        if corpo is not None:
            kwargs["json"] = corpo
        r = getattr(cliente, metodo.lower())(caminho, **kwargs)
        linha += f" {rotulo}={r.status_code}"
    print(linha)

print("\n💡 401 → não identificado · 403 → identificado e barrado")
print("   204/200 → permitido")

## 📝 Exercícios

**E1.** Adicione `ATLAS_TAMANHO_MAX_UPLOAD` à `Config` com validação, e mostre o erro quando alguém definir um valor inválido no `.env`.

**E2.** 🔴 Escreva um teste que falhe se qualquer arquivo `.py` do projeto contiver uma string parecida com senha (`senha=`, `password=`, `secret=`) fora de comentário.

**E3.** Compare `bcrypt`, `hashlib.scrypt` e `sha256` em tempo. Explique por que o mais lento vence.

**E4.** Implemente o *refresh token*: `/auth/refresh` recebe um token de longa duração e devolve um novo token de acesso curto.

**E5.** Corrija o vazamento por tempo do login: rode um hash descartável quando o e-mail não existir, e meça antes e depois.

**E6.** Adicione o claim `escopos: ["produtos:ler", "produtos:escrever"]` ao token e uma dependência `exigir_escopo(nome)`. Compare com o modelo de papéis.

**E7.** Implemente uma lista de revogação (`jti` + conjunto de tokens invalidados) e uma rota `/auth/sair`.

**E8.** Escreva um middleware de limitação de taxa: no máximo 10 requisições por minuto por IP, respondendo `429` com `Retry-After`.

**E9.** Adicione um middleware que rejeite corpos maiores que 1 MB com `413`.

**E10.** Configure o CORS por ambiente: permissivo em desenvolvimento, restrito em produção. Prove com dois valores de `ATLAS_AMBIENTE`.

**E11.** Adicione ordenação por múltiplos campos (`ordenar_por=categoria,-preco`) mantendo a lista branca.

**E12.** Substitua a paginação por offset por paginação por **cursor** e explique qual problema isso resolve em listas grandes.

**E13.** 🔴 Encontre a falha: a rota `GET /produtos/{sku}` deixa qualquer usuário autenticado ver qualquer SKU. Como você implementaria "cada fornecedor só vê os próprios produtos"? (Isto é *Broken Object Level Authorization*, o item nº 1 do OWASP API Top 10.)

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

## 📋 Cola de referência

```python
# ═══ Configuração ═══
class Config(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_prefix="ATLAS_")
    chave_secreta: str = Field(min_length=32)      # 🔴 sem default

@lru_cache
def obter_config() -> Config: return Config()

# .gitignore  →  .env          🔴 o arquivo mais importante
# .env.exemplo →  vai para o Git, sem valores reais
# python -c "import secrets; print(secrets.token_urlsafe(48))"

# ═══ Senha 🔴 ═══
bcrypt.hashpw(senha.encode(), bcrypt.gensalt(rounds=12))
bcrypt.checkpw(senha.encode(), hash_guardado.encode())
# NUNCA sha256/md5. NUNCA guardar em texto. NUNCA logar.

# ═══ JWT ═══
jwt.encode({"sub": email, "papel": p, "exp": ..., "iat": ..., "iss": ...},
           chave, algorithm="HS256")
jwt.decode(token, chave, algorithms=["HS256"], issuer="atlas-api")
# 🔴 ASSINADO, não criptografado — qualquer um LÊ a carga.

# ═══ Autenticação ═══
esquema = OAuth2PasswordBearer(tokenUrl="/auth/token")

def usuario_atual(token: Annotated[str, Depends(esquema)]) -> dict: ...

@roteador.post("/token")
def entrar(f: Annotated[OAuth2PasswordRequestForm, Depends()]): ...
# 🔴 mensagem de erro IDÊNTICA para e-mail e senha errados
# 🔴 401 sempre com headers={"WWW-Authenticate": "Bearer"}

# ═══ Autorização (fábrica de dependências) ═══
def exigir_papel(minimo: str):
    def verificar(usuario: UsuarioDep) -> dict: ...
    return verificar

AdminDep = Annotated[dict, Depends(exigir_papel("admin"))]
# 401 = não sei quem é   ·   403 = sei e não pode

# ═══ Middleware ═══
@app.middleware("http")
async def m(requisicao: Request, proximo):
    r = await proximo(requisicao)
    r.headers["X-Request-ID"] = ...
    return r
# ⚠️ o ÚLTIMO adicionado fica mais perto da rota; CORS é o mais externo

# ═══ CORS ═══
app.add_middleware(CORSMiddleware, allow_origins=[...],
                   allow_credentials=True, allow_methods=[...],
                   allow_headers=["Authorization", "Content-Type"])
# 🔴 ["*"] + credentials = recusado pelo navegador
# ⚠️ CORS protege o USUÁRIO, não a API. curl ignora.

# ═══ Filtros 🔴 ═══
CAMPOS_ORDENAVEIS = ("sku", "nome", "preco")     # LISTA BRANCA
if campo not in CAMPOS_ORDENAVEIS: raise HTTPException(422, ...)
# placeholder protege VALOR; identificador exige lista fechada
```

## ✅ Checklist de saída

- [ ] 🔴 **Nenhum segredo no código**; tudo via `Settings` e `.env`
- [ ] `.env` no `.gitignore`, `.env.exemplo` no repositório
- [ ] A aplicação **não sobe** com configuração inválida
- [ ] 🔴 **Guardo hash de senha, nunca a senha**
- [ ] Uso bcrypt/scrypt/Argon2, nunca sha256
- [ ] Sei que JWT é assinado, **não** criptografado
- [ ] Não coloco dado sensível na carga do token
- [ ] Meus tokens expiram
- [ ] Login devolve erro **idêntico** para usuário e senha errados
- [ ] `401` com `WWW-Authenticate`
- [ ] Distingo **401** (quem é você?) de **403** (você não pode)
- [ ] Uso fábrica de dependências para autorização por papel
- [ ] Escrevo middleware `async` e sem I/O bloqueante
- [ ] Sei que a ordem dos middlewares é invertida e que CORS vai por fora
- [ ] Entendo que CORS protege o navegador, não a API
- [ ] 🔴 **Lista branca para qualquer identificador vindo do usuário**
- [ ] Meus `response_model` não expõem campo interno
- [ ] Li (ou vou ler) o OWASP API Security Top 10

---

### ➡️ Próxima aula

**`06_99_Lista_Exercicios.ipynb`** — A lista completa do módulo e a evolução do Atlas: a API v1 documentada, autenticada e testável.